In [ ]:
# Shape generation worker. Created by `shape generate --scale-mode fabric_spark`.
# The workspace and Lakehouse are filled in when the notebook is created (the Jobs API does not
# pass notebook parameters); the per-run values come from the Spark conf.
from pyspark.sql import SparkSession

spark = SparkSession.builder.getOrCreate()
workspace_id = "__SHAPE_WORKSPACE_ID__"
lakehouse_id = "__SHAPE_LAKEHOUSE_ID__"
job_spec_path = spark.conf.get("spark.shape.jobSpec")
table_prefix = spark.conf.get("spark.shape.tablePrefix", "")
print(f"job spec: {job_spec_path}")

In [ ]:
# Install Shape (and the domain data its schemas use) on the driver and the executors.
import subprocess
import sys

requirements = "__SHAPE_REQUIREMENTS__".split()
r = subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", "--disable-pip-version-check", *requirements],
    capture_output=True,
    text=True,
)
if r.returncode != 0:
    raise RuntimeError(f"pip install failed: {r.stderr[-2000:]}")
print("installed:", ", ".join(requirements))
# Executors need Shape as well: attach it as a library of the notebook's Environment.

In [ ]:
# Read the job spec from the Lakehouse (the cluster is already signed in to OneLake).
from shape.scale import spark_worker

root = f"abfss://{workspace_id}@onelake.dfs.fabric.microsoft.com/{lakehouse_id}"
raw = spark.sparkContext.wholeTextFiles(f"{root}/Files/{job_spec_path}").collect()[0][1]
spec = spark_worker.load_spec(raw)
print(f"{len(spec['row_counts'])} tables, {sum(spec['row_counts'].values()):,} rows")

In [ ]:
# Generate and write the tables as Delta. A failure here fails the run (no success is recorded).
result = spark_worker.run_job(
    spark, spec, workspace_id=workspace_id, lakehouse_id=lakehouse_id, table_prefix=table_prefix
)
spark_worker.check_result(spec, result)
print(result)

In [ ]:
# Record the result next to the spec, then remove the spec.
import json

from notebookutils import mssparkutils

stem = job_spec_path.rsplit(".", 1)[0]
mssparkutils.fs.put(f"{root}/Files/{stem}_result.json", json.dumps(result), True)
mssparkutils.fs.rm(f"{root}/Files/{job_spec_path}")
print("result written")